In [5]:
import numpy as np
import pandas as pd

In [7]:
movies = pd.read_csv('tmdb_5000_movies.csv')
credits = pd.read_csv('tmdb_5000_credits.csv')

In [9]:
movies = movies.merge(credits, on="title")  
# merge 2 dataset based on title so shape 23 means total column 24 (title)....

In [11]:
# genres
# id
# keywords
# title
# overview
# cast
# crew
# will keep that column

movies = movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [13]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [15]:
movies.dropna(inplace=True)
# drop overview raw where value is null , so basically here 3 values are null so we drop it

In [17]:
movies.duplicated().sum() # no any raws are duplicated

0

In [19]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [21]:
# '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'
# this all thing are should convert like,
# ['Action','Adventure','fantasy','scfi']

In [23]:
# helper function to convert this strings into list and get name...
def convert(obj):
     l = []
     for i in ast.literal_eval(obj):
         l.append(i['name'])
     return l

# import ast
# ast.literal_eval
# basically this ast is tranform string into list 

<function ast.literal_eval(node_or_string)>

In [25]:
movies['genres'] = movies['genres'].apply(convert)
# in this apply convert function and get list, after stotr in movies['genres']

In [27]:
# apply same function on keywords...
movies['keywords'] = movies['keywords'].apply(convert)

In [29]:
# to perform opertaion on cast column to get top 3 actor name..
def convert3(obj):
     l = []
     counter = 0
     for i in ast.literal_eval(obj):
         if counter != 3:
             l.append(i['name'])
             counter+=1
         else:
             break
     return l

In [31]:
# apply above function
movies['cast'] = movies['cast'].apply(convert3)

In [33]:
# Work on crew column to fetch only director name
def fetch_director(obj):
     l = []
     for i in ast.literal_eval(obj):
         if i['job'] == 'Director':
             l.append(i['name'])
             break
     return l

In [35]:
movies['crew'] = movies['crew'].apply(fetch_director)

In [37]:
movies['overview'] = movies['overview'].apply(lambda x:x.split())
# convert string into list of overview column so we can use in concatination of other column easily

In [39]:
# remove spaces between word(name) so the Tag will generate and think that is one tag.... (andrew stanton , To andrewstanton)
movies['genres'] = movies['genres'].apply(lambda x:[i.replace(" ","") for i in x]) 
movies['keywords'] = movies['keywords'].apply(lambda x:[i.replace(" ","") for i in x]) 
movies['cast'] = movies['cast'].apply(lambda x:[i.replace(" ","") for i in x]) 
movies['crew'] = movies['crew'].apply(lambda x:[i.replace(" ","") for i in x]) 

In [41]:
# concatinate last 5 column and make one column named Tag
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

In [43]:
# make new dataframe with 3 column movie_id, title, tags
new_df = movies[['movie_id','title','tags']]

In [45]:
# convert tags list into string
new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))

C:\Users\kuldi\AppData\Local\Temp\ipykernel_18740\2694569168.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:" ".join(x))


In [47]:
new_df['tags'][0] # just view how it look

'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization. Action Adventure Fantasy ScienceFiction cultureclash future spacewar spacecolony society spacetravel futuristic romance space alien tribe alienplanet cgi marine soldier battle loveaffair antiwar powerrelations mindandsoul 3d SamWorthington ZoeSaldana SigourneyWeaver JamesCameron'

In [49]:
# convert all string into lower case
new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())

C:\Users\kuldi\AppData\Local\Temp\ipykernel_18740\1214724849.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(lambda x:x.lower())


In [51]:
# apply steming (used for words that are same like[love, loving, loved] so after applying steming it will [love, love, love])

import nltk

from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

In [53]:
# heper function

def stem(text):
    y = []
    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y) # after stemping and aad into list , now convert into string again
    

In [55]:
new_df['tags'] = new_df['tags'].apply(stem) # perform steming on tags and add it in new_df['tags']

C:\Users\kuldi\AppData\Local\Temp\ipykernel_18740\636063548.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  new_df['tags'] = new_df['tags'].apply(stem) # perform steming on tags and add it in new_df['tags']


In [56]:
# perform vectorization on tags , so that easly fine neares movies of main moveis....

from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(max_features=5000, stop_words='english') # stopwords(to, the, are like words) will remove of english language 

In [59]:
vectors = cv.fit_transform(new_df['tags']).toarray() # .toarray() --> convert in numpy array

In [61]:
cv.get_feature_names_out()

array(['000', '007', '10', ..., 'zone', 'zoo', 'zooeydeschanel'],
      dtype=object)

In [63]:
from sklearn.metrics.pairwise import cosine_similarity

# to find similarites betwwen vectors

In [65]:
similarity = cosine_similarity(vectors) # apply library (cosine_similarity) on vectors

In [67]:
similarity[0] # similaritis of first movie with all 4806 movies 
# where first movie with fiest movie similarity always be 1 so we show that 1..

array([1.        , 0.08346223, 0.0860309 , ..., 0.04499213, 0.        ,
       0.        ])

In [69]:
sorted(list(enumerate(similarity[0])), reverse=True, key=lambda x:x[1]) [1:6]

[(1214, 0.28676966733820225),
 (2405, 0.26901379342448517),
 (3728, 0.2605130246476754),
 (507, 0.255608593705383),
 (539, 0.2503866978335957)]

In [71]:
# make function that if i write any movie name it will give me 5 similar movies of that

def recommend(movie):
    movie_index = new_df[new_df['title'] == movie].index[0]
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x:x[1]) [1:6]

    for i in movies_list:
        print(new_df.iloc[i[0]].title)
        
    

In [73]:
recommend('Avatar')

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [75]:
import pickle

In [77]:
pickle.dump(new_df, open('movies.pkl','wb'))

In [79]:
pickle.dump(similarity, open('similarity.pkl','wb'))